# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [ ]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets
from tqdm import tqdm

### Data Preparation

In [197]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

Repo card metadata block was not found. Setting CardData to empty.




Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [355]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

In [199]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [200]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

In [201]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [202]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [203]:
for batch in val_loader:
    break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        token_type_ids=batch["token_type_ids"],
    )

print("\nPrediction (probs):", torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

**Task 1 (1 point)**

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [204]:
val_loader = torch.utils.data.DataLoader(
    val_set,
    batch_size=64,
    shuffle=False,
    collate_fn=transformers.default_data_collator,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)

In [205]:
device = torch.device("cuda")
model.to(device)
model.eval();

In [206]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [207]:
correct = 0
total = 0

amp_ctx = torch.amp.autocast(device_type="cuda", dtype=torch.float16)

with torch.inference_mode():
    for batch in tqdm(val_loader):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        
        token_type_ids = batch.get("token_type_ids")
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device, non_blocking=True)
            
        labels = batch["labels"].to(device, non_blocking=True)

        with amp_ctx:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
            preds = outputs.logits.argmax(dim=-1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"Validation accuracy: {accuracy:.6f}")

100%|██████████| 632/632 [02:18<00:00,  4.55it/s]

Validation accuracy: 0.908360


In [208]:
assert 0.9 < accuracy < 0.91

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

**Task 2 (4 points)**
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

Let's fine-tune deberta-v3-small model:

In [233]:
base_model_name = 'microsoft/deberta-v3-small'

model = transformers.AutoModelForSequenceClassification.from_pretrained(base_model_name, num_labels=2)
tokenizer = transformers.AutoTokenizer.from_pretrained(base_model_name, use_fast=False)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [234]:
model = model.to(device)

In [235]:
from sklearn.metrics import accuracy_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    
    return {"accuracy": acc}

In [236]:
training_args = transformers.TrainingArguments(
    output_dir="./deberta_v3_small_qqp",
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=100,
    report_to="none",
    save_strategy="steps",
    save_steps=2000,
    num_train_epochs=2,                   
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=4,
)

In [237]:
trainer = transformers.Trainer(
    model=model,
    args=training_args,
    train_dataset=qqp_preprocessed["train"],
    eval_dataset=qqp_preprocessed["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

/tmp/ipykernel_37/2789285030.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = transformers.Trainer(


In [238]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [239]:
train_output = trainer.train()
print(train_output)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Step,Training Loss,Validation Loss,Accuracy
500,0.575400,0.538493,0.705540
1000,0.535600,0.510513,0.729805
1500,0.500600,0.503738,0.758447
2000,0.482700,0.471973,0.765397
2500,0.476300,0.467135,0.767005
3000,0.446900,0.433132,0.786569
3500,0.452300,0.422994,0.796438
4000,0.443100,0.472548,0.776799
4500,0.437200,0.412459,0.797626
5000,0.418800,0.408554,0.802993


TrainOutput(global_step=22742, training_loss=0.37096047898352, metrics={'train_runtime': 10283.4583, 'train_samples_per_second': 70.763, 'train_steps_per_second': 2.212, 'total_flos': 2.4099724986107904e+16, 'train_loss': 0.37096047898352, 'epoch': 2.0})


In [308]:
model_dir = "/kaggle/working/deberta_v3_small_qqp/checkpoint-22742"
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
tokenizer = transformers.AutoTokenizer.from_pretrained(model_dir)

In [263]:
val_loader = torch.utils.data.DataLoader(
    val_set,
    batch_size=64,
    shuffle=False,
    collate_fn=transformers.default_data_collator,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)

correct = 0
total = 0

model.eval()

amp_ctx = torch.amp.autocast(device_type="cuda", dtype=torch.float16)

with torch.inference_mode():
    for batch in tqdm(val_loader):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        
        token_type_ids = batch.get("token_type_ids")
        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(device, non_blocking=True)
            
        labels = batch["labels"].to(device, non_blocking=True)

        with amp_ctx:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
            preds = outputs.logits.argmax(dim=-1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"Validation accuracy: {accuracy:.6f}")

100%|██████████| 632/632 [01:31<00:00,  6.94it/s]

Validation accuracy: 0.853450


### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

**Task 3 (1 point)**
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

In [351]:
model_dir = "/kaggle/working/deberta_v3_small_qqp/checkpoint-22742"
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
tokenizer = transformers.AutoTokenizer.from_pretrained(model_dir)

In [348]:
model.to(device)
model.eval()

def build_train_question_pool(dataset) -> list[str]:
    text1 = dataset["text1"]
    text2 = dataset["text2"]
    pool = list(set(text1) | set(text2))
    
    return pool

train_questions_pool = build_train_question_pool(qqp_preprocessed["train"])
print(f"Unique train questions: {len(train_questions_pool):}")


Unique train questions: 493874


In [340]:
def score_query(query: str,candidates: list[str], batch_size: int = 128):
    amp_ctx = torch.amp.autocast(device_type="cuda", dtype=torch.float16)

    probs_ones = np.empty(len(candidates), dtype=np.float32)

    with torch.inference_mode():
        for start in tqdm(range(0, len(candidates), batch_size)):
            end = min(start + batch_size, len(candidates))
            batch_cands = candidates[start:end]

            enc = tokenizer(
                [query] * len(batch_cands),
                batch_cands,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt",
            )

            enc = {k: v.to(device, non_blocking=True) for k, v in enc.items()}

            with amp_ctx:
                out = model(**enc).logits
                batch_probs = torch.softmax(out, dim=-1)[:, 1]

            probs_ones[start:end] = batch_probs.detach().float().cpu().numpy()

    return probs_ones

def find_duplicates(
    query: str,
    k: int = 5,
    candidate_pool: list[str] = None,
    limit_candidates: int = None,
):
    if candidate_pool is None:
        candidate_pool = train_questions_pool

    if limit_candidates is not None and limit_candidates < len(candidate_pool):
        pool = candidate_pool[:limit_candidates]
    else:
        pool = candidate_pool

    probs = score_query(query, pool, batch_size=256)
    top_idx = np.argsort(-probs)[:k]

    return [(pool[i], float(probs[i])) for i in top_idx]

def show_duplicates(query: str, k: int = 5, limit_candidates: int = 50000):
    print("\n" + "=" * 100)
    print("Query:")
    print(query)
    print("-" * 100)
    results = find_duplicates(query, k=k, limit_candidates=limit_candidates)
    for rank, (cand, p) in enumerate(results, 1):
        print(f"{rank}.  p_dup={p:0.4f}  |  {cand}")


Unique train questions: 493874


In [344]:
show_duplicates("How do I control my horny emotions?", k=5)


Query:
How do I control my horny emotions?
----------------------------------------------------------------------------------------------------


100%|██████████| 196/196 [01:44<00:00,  1.88it/s]

1.  p_dup=0.9978  |  How can someone overcome servere social anxiety?
2.  p_dup=0.9976  |  HI MY ZENFONE SELFIE STRUCK ON FASTBOOT MODE?
3.  p_dup=0.9974  |  Does our mind control our emotions?
4.  p_dup=0.9966  |  How do I learn how to use self control and get over illogical jealousy?
5.  p_dup=0.9963  |  How do you control your horniness?


In [345]:
show_duplicates("Can anyone recommend good online course on python programming?", k=5)


Query:
Can anyone recommend good online course on python programming?
----------------------------------------------------------------------------------------------------


100%|██████████| 196/196 [01:48<00:00,  1.81it/s]

1.  p_dup=0.9983  |  What are the best learning sites for Python?
2.  p_dup=0.9966  |  What are some good books or online tutorials for learning Python from basics?
3.  p_dup=0.9915  |  What are some resources to learn Advanced Python 3?
4.  p_dup=0.9885  |  Are there any good game programming colleges in India?
5.  p_dup=0.9809  |  I want to learn and eventually master Python. Where do I start?


In [368]:
show_duplicates("Have you ever climbed a mountain?", k=5)


Query:
Have you ever climbed a mountain?
----------------------------------------------------------------------------------------------------


100%|██████████| 196/196 [01:41<00:00,  1.93it/s]

1.  p_dup=0.1315  |  What does it feel like to climb Mt. Everest?
2.  p_dup=0.0130  |  I HAVE TWO WHEELER LICENSE FROM WEST BENGAL.CAN I DRIVE BIKE THROUHOUT INDIA?
3.  p_dup=0.0106  |  HI MY ZENFONE SELFIE STRUCK ON FASTBOOT MODE?
4.  p_dup=0.0093  |  CAN ANY ONE TELL ME WHERE I SHOULD STUDY AVIATION COURSE?
5.  p_dup=0.0066  |  WHAT ARE DIFFERENT M.B.A BRANCHES OTHER THAN TRADITIONAL ONES WITH GOOD PACKAGE IN INDIA?


In [361]:
show_duplicates("Quora is definitely the best source of information", k=5)


Query:
Quora is definitely the best source of information
----------------------------------------------------------------------------------------------------


100%|██████████| 196/196 [01:45<00:00,  1.85it/s]

1.  p_dup=0.9833  |  What is Quora most useful for?
2.  p_dup=0.9809  |  FISICA_TEORICA Google?
3.  p_dup=0.9777  |  Which are the best Quora answers one must read?
4.  p_dup=0.9451  |  Is Quora your best source for knowledge?
5.  p_dup=0.9449  |  What is Quora all about.?


In [364]:
show_duplicates("Oh no, anyway", k=5)


Query:
Oh no, anyway
----------------------------------------------------------------------------------------------------


100%|██████████| 196/196 [01:36<00:00,  2.03it/s]

1.  p_dup=0.0274  |  I don't have one. BUT, FUCK DONALD TRUNP?
2.  p_dup=0.0077  |  What is your weird habit that is useless?
3.  p_dup=0.0076  |  My
4.  p_dup=0.0059  |  HH
5.  p_dup=0.0049  |  What are some of your weird habits?


### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

**Bonus Task 1 (0.5 point)**
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

In [ ]:
<A whole lot of YOUR CODE HERE>

### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

**Bonus Task 2 (1.5 points)**
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part

In [ ]:
<A whole lot of YOUR CODE HERE>